In [1]:
import awkward as ak
import json

In [2]:
df = ak.from_parquet('SM_23Sep22_full/merged_nominal.parquet')

In [3]:
with open('SM_23Sep22_full/summary.json') as f:
    summary = json.load(f)

In [4]:
proc_ids = summary['sample_id_map']
print(proc_ids)

{'Data': 0, 'DataDrivenGJets': -99, 'DiPhoton': 18, 'GJets_HT-100To200': 9, 'GJets_HT-200To400': 10, 'GJets_HT-400To600': 11, 'GJets_HT-40To100': 8, 'GJets_HT-600ToInf': 12, 'HHggTauTau': 20, 'HHggWW_dileptonic': 21, 'HHggWW_semileptonic': 22, 'HHggbb': 19, 'TTGG': 15, 'TTGamma': 14, 'TTJets': 13, 'VBFH_M125': 6, 'VH_M125': 7, 'WGamma': 16, 'ZGamma': 17, 'ggH_M125': 5, 'ttHH_ggTauTau': 4, 'ttHH_ggWW': 3, 'ttHH_ggbb': 2, 'ttH_M125': 1}


In [5]:
signal = ['ttHH_ggbb', 'ttHH_ggWW', 'ttHH_ggTauTau']
nonres = ['DiPhoton', 'TTGG', 'TTGamma', 'TTJets', 'WGamma', 'ZGamma', 'DataDrivenGJets']
res = ['VBFH_M125', 'VH_M125', 'ggH_M125', 'ttH_M125']
hh = ['HHggbb', 'HHggTauTau', 'HHggWW_dileptonic', 'HHggWW_semileptonic']
gjets = ['GJets_HT-40To100', 'GJets_HT-100To200', 'GJets_HT-200To400', 'GJets_HT-400To600', 'GJets_HT-600ToInf']

In [6]:
def group_mask(df, proc_ids, group):
    tmp_mask = df.process_id==-857 #Not -999 incase some error has occured
    for proc in group:
        tmp_mask = tmp_mask | (df.process_id==proc_ids[proc])
    return tmp_mask

In [7]:
nonres_mask = group_mask(df, proc_ids, nonres)
res_mask = group_mask(df, proc_ids, res)
signal_mask = group_mask(df, proc_ids, signal)
data_mask = df.process_id ==proc_ids['Data']
SR1_mask = df.mva_score>=0.9882 #SR from MC as nonres bkg
SR2_mask = ((df.mva_score>=0.949234)&(~SR1_mask))

In [14]:
def yield_counter(df, name, mask_1, mask_2):
    proc_mask = df.process_id == proc_ids[name]
    min_edge = 100
    low_edge = 120
    high_edge = 130
    max_edge = 180
    low_mask = (df.Diphoton_mass >= min_edge) & (df.Diphoton_mass <= low_edge)
    peak_mask = (df.Diphoton_mass > low_edge) & (df.Diphoton_mass < high_edge)
    high_mask = (df.Diphoton_mass >= high_edge) & (df.Diphoton_mass <= max_edge)
    low_count_1 = ak.sum(df[proc_mask & low_mask & mask_1].weight_central)
    peak_count_1 = ak.sum(df[proc_mask & peak_mask & mask_1].weight_central)
    high_count_1 = ak.sum(df[proc_mask & high_mask & mask_1].weight_central)
    low_count_2 = ak.sum(df[proc_mask & low_mask & mask_2].weight_central)
    peak_count_2 = ak.sum(df[proc_mask & peak_mask & mask_2].weight_central)
    high_count_2 = ak.sum(df[proc_mask & high_mask & mask_2].weight_central)
    #print('Num events in the low bin: {}'.format(low_count))
    #print('Num events in the peak bin: {}'.format(peak_count))
    #print('Num events in the high bin: {}'.format(high_count))
    #print('----------------------------------')
    #print(name+' low & {:.4f} & {:.4f} & {:.4f}'.format(low_count_1, low_count_2, low_count_1+low_count_2))
    #print(name+' peak & {:.4f} & {:.4f}& {:.4f}'.format(peak_count_1, peak_count_2, peak_count_1+peak_count_2))
    #print(name+' high & {:.4f} & {:.4f}& {:.4f}'.format(high_count_1, high_count_2, high_count_1+high_count_2))
    full_count_1 = low_count_1 + peak_count_1 + high_count_1
    full_count_2 = low_count_2 + peak_count_2 + high_count_2
    name = name.replace('_','-')
    print(name+' & {:.4f} & {:.4f} & {:.4f} \\\\hline'.format(full_count_1, full_count_2, full_count_1+full_count_2))

In [9]:
def sidebands_only(df):
    min_edge = 100
    low_edge = 120
    high_edge = 130
    max_edge = 180
    low_mask = (df.Diphoton_mass >= min_edge) & (df.Diphoton_mass <= low_edge)
    high_mask = (df.Diphoton_mass >= high_edge) & (df.Diphoton_mass <= max_edge)
    low_count = ak.sum(df[low_mask].weight_central)
    high_count = ak.sum(df[high_mask].weight_central)
    #print('Num events in the low bin: {}'.format(low_count))
    #print('Num events in the high bin: {}'.format(high_count))
    #print('----------------------------------')
    print('Num events in the low bin: {:.4f}'.format(low_count))
    print('Num events in the high bin: {:.4f}'.format(high_count))

In [15]:
print('Process & SR1 & SR2 & Total')
for sig in signal:
    yield_counter(df, sig, SR1_mask, SR2_mask)

Process & SR1 & SR2 & Total
ttHH-ggbb & 0.0528 & 0.0299 & 0.0827 \\hline
ttHH-ggWW & 0.0126 & 0.0122 & 0.0249 \\hline
ttHH-ggTauTau & 0.0048 & 0.0036 & 0.0085 \\hline


In [16]:
print('Process & SR1 & SR2 & Total')
for bkg in res:
    yield_counter(df, bkg, SR1_mask, SR2_mask)

Process & SR1 & SR2 & Total
VBFH-M125 & 0.0005 & 0.0030 & 0.0035 \\hline
VH-M125 & 0.0271 & 0.1786 & 0.2057 \\hline
ggH-M125 & 0.0154 & 0.2324 & 0.2478 \\hline
ttH-M125 & 5.1625 & 12.0580 & 17.2205 \\hline


In [17]:
print('Process & SR1 & SR2 & Total')
for bkg in hh:
    yield_counter(df, bkg, SR1_mask, SR2_mask)

Process & SR1 & SR2 & Total
HHggbb & 0.0201 & 0.0979 & 0.1180 \\hline
HHggTauTau & 0.0022 & 0.0079 & 0.0100 \\hline
HHggWW-dileptonic & 0.0010 & 0.0035 & 0.0045 \\hline
HHggWW-semileptonic & 0.0018 & 0.0085 & 0.0103 \\hline


In [18]:
print('Process & SR1 & SR2 & Total')
for bkg in nonres:
    yield_counter(df, bkg, SR1_mask, SR2_mask)

Process & SR1 & SR2 & Total
DiPhoton & 3.7235 & 30.3898 & 34.1134 \\hline
TTGG & 3.5990 & 12.9572 & 16.5562 \\hline
TTGamma & 9.8711 & 70.6495 & 80.5206 \\hline
TTJets & 1.4813 & 41.7329 & 43.2142 \\hline
WGamma & 0.9906 & 3.7506 & 4.7412 \\hline
ZGamma & -0.2523 & 1.9572 & 1.7050 \\hline
DataDrivenGJets & 0.0000 & 27.5711 & 27.5711 \\hline
